In [1]:
from langchain.agents import create_agent;
from langchain.tools import tool
from langchain_ollama import ChatOllama
@tool
def get_weather(city:str) -> str:
    """Get weather for a city."""
    return f"The weather in {city} is always breezy."

agent = create_agent(
    model=ChatOllama(model='qwen2.5:3b'),
    tools=[get_weather],
)
for chunk in agent.stream({'messages':{'role':'user', 'content':'What is the weather in pune?'}}, stream_mode='updates',):
    # print(chunk)
    for step, data in chunk.items():
        print(f'step: {step}')
        print(f'content: {data['messages'][-1].content_blocks}')

step: model
content: [{'type': 'tool_call', 'id': '552fd0f1-c661-4965-bdb4-b4368f61a16e', 'name': 'get_weather', 'args': {'city': 'pune'}}]
step: tools
content: [{'type': 'text', 'text': 'The weather in pune is always breezy.'}]
step: model
content: [{'type': 'text', 'text': 'The weather in Pune is always breezy. It seems to be a light and refreshing wind. Is there anything else you would like to know?'}]


In [2]:
for token,metadata in agent.stream({'messages':{'role':'user', 'content':'What is the weather in Pune?'}}, stream_mode='messages'):
    print(f'source: {metadata['langgraph_node']}')
    print(f'token: {token.content_blocks}\n')
    # print(chunk)

source: model
token: [{'type': 'tool_call_chunk', 'id': 'd3d9ba06-7725-4d3a-9ff9-f727c9af3ef8', 'name': 'get_weather', 'args': '{"city": "Pune"}'}]

source: model
token: []

source: model
token: []

source: tools
token: [{'type': 'text', 'text': 'The weather in Pune is always breezy.'}]

source: model
token: [{'type': 'text', 'text': 'The'}]

source: model
token: [{'type': 'text', 'text': ' weather'}]

source: model
token: [{'type': 'text', 'text': ' in'}]

source: model
token: [{'type': 'text', 'text': ' Pune'}]

source: model
token: [{'type': 'text', 'text': ' is'}]

source: model
token: [{'type': 'text', 'text': ' always'}]

source: model
token: [{'type': 'text', 'text': ' bree'}]

source: model
token: [{'type': 'text', 'text': 'zy'}]

source: model
token: [{'type': 'text', 'text': '.'}]

source: model
token: [{'type': 'text', 'text': ' Is'}]

source: model
token: [{'type': 'text', 'text': ' there'}]

source: model
token: [{'type': 'text', 'text': ' anything'}]

source: model
token:

In [3]:
from langgraph.config import get_stream_writer
model = ChatOllama(model='qwen2.5:3b')
@tool
def get_population(city:str)->str:
    """Get population data for city."""
    writer = get_stream_writer()
    writer(f'Fetching population data for {city}')
    writer(f'Fetched population data for {city}')
    return f"The population of {city} is 20lakhs"
agent = create_agent(
    model=model,
    tools=[get_population]
)
for chunk in agent.stream({'messages':{'role':'user','content':'what\'s the population of Pune?'}},stream_mode='custom'):
    print(chunk)

Fetching population data for Pune
Fetched population data for Pune


In [4]:
for stream_mode,chunk in agent.stream({'messages':{'role':'user','content':'What\'s the population of pune?'}}, stream_mode=['updates','custom']):
    print(f'mode: {stream_mode}')
    print(f'chunk: {chunk}\n')

mode: updates
chunk: {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-30T17:34:17.7267053Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3312052600, 'load_duration': 154698600, 'prompt_eval_count': 152, 'prompt_eval_duration': 781520700, 'eval_count': 21, 'eval_duration': 2340142000, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019c0ff8-438c-7893-a895-6790f5b0350c-0', tool_calls=[{'name': 'get_population', 'args': {'city': 'pune'}, 'id': 'af4c318b-a925-4c87-8a05-3602c2fd4a28', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 21, 'total_tokens': 173})]}}

mode: custom
chunk: Fetching population data for pune

mode: custom
chunk: Fetched population data for pune

mode: updates
chunk: {'tools': {'messages': [ToolMessage(content='The population of pune is 20lakhs', name='get_population', id='14d76743-1204-475e-8448-a9cd737

In [5]:
from langchain.agents.middleware import AgentState, after_agent
from pydantic import BaseModel
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any, Literal

class ResponseSafety(BaseModel):
    """Evaluate responses as safe or unsafe."""
    safety_decision: Literal['safe', 'unsafe']

safety_checker = ChatOllama(model='qwen2.5:3b')

@after_agent(can_jump_to=['end'])
def safety_guardrail(state:AgentState, runtime:Runtime)->dict[str,Any]|None:
    """Model Based Guardrail: Use an LLM to evaluate response safety."""
    stream_writer = get_stream_writer()
    if not state['messages']:
        return None
    last_message = state['messages'][-1]
    if not isinstance(last_message, AIMessage):
        return None
    
    model_with_tools = safety_checker.bind_tools([ResponseSafety], tool_choice='any')
    result = model_with_tools.invoke(
        [{'role':'system',
          'content':'Evaluate the messages received from AI as safe or unsafe.'},
          {'role':'user',
           'content':f'AI response: {last_message.text}'}]
    )
    stream_writer(result)
    tool_call = result.tool_calls[0]
    if tool_call['args'].get('safety_decision') == 'unsafe':
        last_message.content = "That response is not safe. Please rephrase your request."
    return None

In [6]:
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
agent = create_agent(
    model=model,
    tools=[get_weather, get_population],
    middleware=[safety_guardrail]
)
def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end='|')
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    
def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")

In [9]:
query = {'role':'user', 'content':'What\'s the weather in Goa?'}
for stream_mode, data in agent.stream(
    {'messages': [query]},
    stream_mode=['messages', 'updates', 'custom']
):
    if stream_mode == 'messages':
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == 'updates':
        print('stream_mode: updates')
        for source, update in data.items():
            if source in ('model', 'tools'):
                _render_completed_message(update['messages'][-1])
    if stream_mode == 'custom':
        print('stream_mode: custom')
        print(f'Tool calls: {data.tool_calls}')

[{'name': 'get_weather', 'args': '{"city": "Goa"}', 'id': '887ba3fa-20b0-4030-9da3-d5f94061bd2f', 'index': None, 'type': 'tool_call_chunk'}]
stream_mode: updates
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Goa'}, 'id': '887ba3fa-20b0-4030-9da3-d5f94061bd2f', 'type': 'tool_call'}]
stream_mode: updates
Tool response: [{'type': 'text', 'text': 'The weather in Goa is always breezy.'}]
The| current| weather| in| Goa| is| described| as| bree|zy|.| Please| note| that| this| information| might| be| outdated|,| as| the| weather| can| change| quickly|.

|Is| there| anything| else| you| would| like| to| know| about| Goa|?|stream_mode: updates
[{'name': 'ResponseSafety', 'args': '{"response": "AI response: The current weather in Goa is described as breezy. Please note that this information might be outdated, as the weather can change quickly.\\n\\nIs there anything else you would like to know about Goa?"}', 'id': '6a831bff-fdcb-4b79-8d81-484e6533188b', 'index': None, 'type': 'tool_call_